# Day 5 — First Classification Model (Baseline)

**Supervised learning:** predict `Career` from student features using **logistic regression**.

**Goal:** verify the pipeline end-to-end and record **test-set** metrics (not train).

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)


## Setup

Run with cwd `notebooks/day05/`. Training and metrics helpers are in `src/classification.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found. cwd should be notebooks/dayXX/.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification import (
    evaluate_multiclass,
    fit_baseline,
    plot_confusion_matrix,
    save_baseline_run,
)
from src.preprocessing import (
    load_cleaned_table,
    prepare_train_test_bundle,
    save_artifacts,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Rebuild the Day 3 train/test split

Same `random_state=42` as Day 3 so rows line up across notebooks.

- `X_*` = scaled + one-hot **features**
- `y_*` = integer labels; `career_classes` maps index → career name

In [ ]:
df = load_cleaned_table(PROJECT_ROOT)
bundle = prepare_train_test_bundle(df, test_size=0.2, random_state=42)
# Refresh saved preprocessor so Day 4+ stay in sync
save_artifacts(bundle, PROJECT_ROOT)

X_train, X_test = bundle["X_train"], bundle["X_test"]
y_train, y_test = bundle["y_train"], bundle["y_test"]
class_names = bundle["career_classes"]
X_train.shape, X_test.shape, len(class_names)

## 2. Fit logistic regression baseline

Multinomial logistic regression with L2 regularization (sklearn default) — fast, interpretable, outputs probabilities.

In [ ]:
# Learn weights from training rows only
model = fit_baseline(X_train, y_train)
model

## 3. Evaluate on the **test** set only

- **Accuracy:** overall correct predictions
- **F1 macro:** each career weighted equally (flags weak rare classes)
- **F1 weighted:** dominated by frequent careers

In [ ]:
metrics = evaluate_multiclass(model, X_test, y_test, class_names)
print(f"Test accuracy: {metrics['accuracy']:.4f}")
print(f"Test F1 (macro): {metrics['f1_macro']:.4f}")
print(f"Test F1 (weighted): {metrics['f1_weighted']:.4f}")
print()
# Per-class precision / recall / F1 — paste this block into your report
print(metrics["classification_report"])

## 4. Confusion matrix (test)

Rows = **true** career, columns = **predicted**. Diagonal = correct top-1 predictions.

In [ ]:
import numpy as np

cm = np.asarray(metrics["confusion_matrix"])
fig = plot_confusion_matrix(cm, class_names)
plt.show()

## 5. Probabilities for one test student (preview for Day 10)

`predict()` = single best career. `predict_proba()` = full ranking vector (sums to ~1).

In [ ]:
proba = metrics["y_proba"][0]  # first student in X_test order
top = pd.Series(proba, index=class_names).sort_values(ascending=False).head(5)
top

## 6. Save metrics, report, figure, and model

| Output | Location |
|--------|----------|
| Summary JSON | `outputs/metrics/day05_baseline_metrics.json` |
| Full report | `outputs/metrics/day05_baseline_classification_report.txt` |
| Confusion plot | `outputs/figures/day05_baseline_confusion_matrix.png` |
| Fitted model | `models/day05_baseline_logistic.joblib` |

In [ ]:
paths = save_baseline_run(model, metrics, class_names, PROJECT_ROOT)
paths

## Checkpoint (for your report)

1. Why use logistic regression before trees or neural nets?
2. Why report macro F1 as well as accuracy?
3. Paste **your** test metrics from `outputs/metrics/day05_baseline_metrics.json` — do not copy example numbers from slides.